# 01 — Exploratory Data Analysis

Quick tour of the 1994/95 CPS "census-bureau" dataset. This notebook reproduces
and extends what `src/eda.py` does, with interactive outputs inline.

**Goal of the project**
1. Train a classifier to predict whether a person earns more than \$50k/yr
   so the retail client can size and target income-tier marketing campaigns.
2. Build a population segmentation so the client can tailor creative, offers
   and channels to meaningful life-stage / economic groups.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from data_utils import (load_raw, drop_exact_duplicates,
                        LABEL_COL, WEIGHT_COL, NUMERIC_COLS, CATEGORICAL_COLS)

df = drop_exact_duplicates(load_raw(ROOT / "data" / "census-bureau.data"))
print("Shape:", df.shape)
df.head()

## Class balance

Important: the data comes from a stratified sample, so rows carry a
**weight** that tells us the relative frequency in the true population.
We always report class balance and evaluation metrics *both* weighted and
unweighted.

In [ ]:
unw = df[LABEL_COL].mean()
w   = (df[LABEL_COL] * df[WEIGHT_COL]).sum() / df[WEIGHT_COL].sum()
print(f"Unweighted positive rate: {unw:.4f}")
print(f"Weighted positive rate  : {w:.4f}")
df[LABEL_COL].map({0:'<=50K', 1:'>50K'}).value_counts()

## Missingness

The `?` sentinel was converted to NaN in `load_raw`. A handful of columns are ~50% missing.

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].head(10)

## Figures generated by `src/eda.py`

In [ ]:
from IPython.display import Image
for p in sorted((ROOT / "figures").glob("0[1-5]*.png")):
    display(Image(filename=str(p)))

## Key observations

* **Severe class imbalance** (~6% positive) — accuracy is the wrong metric; we
  will use ROC-AUC, PR-AUC, and F1 at a tuned threshold.
* **`weight` is a sampling weight, not a feature** — drop it from X, pass it
  as `sample_weight` to models and metrics.
* **Duplicates**: ~3,200 rows are exact duplicates; we de-duplicate before
  splitting to avoid leakage.
* **`Not in universe`** is a valid "this question doesn't apply" code, not a
  data-quality issue. We keep it as a level.
* **Four migration columns are ~50% missing** because they are only collected
  for people who moved. We impute with `missing` and let the tree model use
  missingness as signal.
